# Differential Testing
Identifying functional differences between two code candidates by generating different inputs

In [1]:
from common.coevolution import lcb_dataset
problems = lcb_dataset.load_code_generation_dataset(
    release_version="release_v6",
    start_date="2025-03-01",
    end_date="2025-05-10",
    difficulty=lcb_dataset.Difficulty.HARD,
)

/Users/kaushitha/Documents/APR/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-08 09:25:41.720 | INFO     | common.coevolution.lcb_dataset:load_code_generation_dataset:127 - Using 'test' split of the dataset.
2025-12-08 09:26:04.759 | INFO     | common.coevolution.lcb_dataset:load_code_generation_dataset:141 - Filtered problems by difficulty: hard
2025-12-08 09:26:04.893 | INFO     | common.coevolution.lcb_dataset:load_code_generation_dataset:151 - Loaded 37 problems


In [2]:
problem_id = 'arc194_c'
problem = next((p for p in problems if p.question_id == problem_id), None)
if problem is None:
    raise ValueError(f"Problem with ID {problem_id} not found.")

# Candidate Code Solutions

In [3]:
C0_snippet = """
class Solution:
    def sol(self, input_str: str) -> str:
        data = list(map(int, input_str.split()))
        it = iter(data)
        N = next(it)
        A = [next(it) for _ in range(N)]
        B = [next(it) for _ in range(N)]
        C = [next(it) for _ in range(N)]
        # Strategy: flip some 1->0 that reduce future costs before doing 0->1 that raise them.
        ones_sum = sum(C[i] for i in range(N) if A[i] == 1)
        idxs_to_zero = [i for i in range(N) if A[i] == 1 and B[i] == 0]
        idxs_to_one = [i for i in range(N) if A[i] == 0 and B[i] == 1]
        # Flip those 1->0 in descending C to maximize immediate reduction early.
        idxs_to_zero.sort(key=lambda i: -C[i])
        total = 0
        for i in idxs_to_zero:
            A[i] = 0
            ones_sum -= C[i]
            total += ones_sum
        # Then flip 0->1 in ascending C to minimize increments when many ones exist
        idxs_to_one.sort(key=lambda i: C[i])
        for i in idxs_to_one:
            A[i] = 1
            ones_sum += C[i]
            total += ones_sum
        return str(total)
"""

C1_snippet = """class Solution:
    def sol(self, input_str: str) -> str:
        # Input is provided as a single string.
        data = list(map(int, input_str.split()))
        it = iter(data)
        try:
            N = next(it)
        except StopIteration:
            return "0"
        A = [next(it) for _ in range(N)]
        B = [next(it) for _ in range(N)]
        C = [next(it) for _ in range(N)]

        # For the given tests, N is up to 20 in the failing case.
        # Implement exact DP over subsets of all positions (not only mismatches) to be safe.
        # We will only flip positions where A[i] != B[i], but represent state as bitmask of which positions have been flipped.
        mismatches = [i for i in range(N) if A[i] != B[i]]
        m = len(mismatches)

        # If no mismatches, cost is 0
        if m == 0:
            return "0"

        # Use DP over subsets of mismatches (m <= 20 in tests)
        # Map j in [0,m) to pos = mismatches[j]
        pos = mismatches
        Apos = [A[p] for p in pos]
        Cpos = [C[p] for p in pos]

        # initial total sum of A * C over all positions (including matching ones)
        init_sum = sum(A[i] * C[i] for i in range(N))

        # For flipping a mismatch j, the change to total sum is delta_j = (1 - 2*Apos[j]) * Cpos[j]
        delta = [(1 - 2 * Apos[j]) * Cpos[j] for j in range(m)]

        total_states = 1 << m
        # Precompute sum_delta for states
        sum_delta = [0] * total_states
        for state in range(1, total_states):
            lsb = state & -state
            j = (lsb.bit_length() - 1)
            prev = state ^ lsb
            sum_delta[state] = sum_delta[prev] + delta[j]

        INF = 10**30
        dp = [INF] * total_states
        dp[0] = 0
        for state in range(total_states):
            if dp[state] == INF:
                continue
            cur_sum = init_sum + sum_delta[state]
            # try flipping any not-yet-flipped mismatch j
            rem = (~state) & (total_states - 1)
            s = rem
            while s:
                lsb = s & -s
                j = (lsb.bit_length() - 1)
                next_state = state | lsb
                # cost of this operation uses A after flip, so it's cur_sum + delta_j
                cost_of_op = cur_sum + delta[j]
                new_cost = dp[state] + cost_of_op
                if new_cost < dp[next_state]:
                    dp[next_state] = new_cost
                s ^= lsb

        ans = dp[total_states - 1]
        return str(ans)
"""

In [4]:
# Creating code individuals and populations
from common.coevolution.core.individual import CodeIndividual
from common.coevolution.core.population import CodePopulation
from common.coevolution.core.interfaces import Operations


c0_ind = CodeIndividual(
    snippet=C0_snippet,
    probability=0.75,
    creation_op=Operations.INITIAL,
    generation_born=0,
    parent_ids=[],
)

c1_ind = CodeIndividual(
    snippet=C1_snippet,
    probability=0.75,
    creation_op=Operations.INITIAL,
    generation_born=0,
    parent_ids=[],
)


code_pop = CodePopulation(individuals=[c0_ind, c1_ind], generation=0)

2025-12-08 09:33:32.573 | DEBUG    | common.coevolution.core.individual:__init__:42 - Created new <CodeIndividual id=C0 gen=0 op=initial prob=0.75>
2025-12-08 09:33:32.574 | DEBUG    | common.coevolution.core.individual:__init__:42 - Created new <CodeIndividual id=C1 gen=0 op=initial prob=0.75>
2025-12-08 09:33:32.575 | DEBUG    | common.coevolution.core.interfaces:__init__:479 - Initialized CodePopulation with 2 individuals, gen 0


In [5]:
problem.public_test_cases

[LCBTest(input='4\n0 1 1 1\n1 0 1 0\n4 6 2 9', output='16', testtype=<TestType.STDIN: 'stdin'>),
 LCBTest(input='5\n1 1 1 1 1\n1 1 1 1 1\n1 1 1 1 1', output='0', testtype=<TestType.STDIN: 'stdin'>),
 LCBTest(input='20\n1 1 1 1 0 0 1 1 0 0 0 1 0 1 0 1 1 0 1 0\n0 0 0 1 1 1 0 1 1 0 0 0 0 0 0 1 0 1 0 0\n52 73 97 72 54 15 79 67 13 55 65 22 36 90 84 46 1 2 27 8', output='2867', testtype=<TestType.STDIN: 'stdin'>)]

In [ ]:
from common.sandbox import create_safe_test_environment

sandbox = create_safe_test_environment()
sandbox.execute_code()

# DET Prompt